## Bertopic

Here we just follow [BertTopic's QuickStart](https://maartengr.github.io/BERTopic/getting_started/quickstart/quickstart.html) to provide an overview of what a classical topic modeling approach would do for our subset of data.

In [1]:
import os
import json

from bertopic import BERTopic

import numpy as np
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from label_studio_sdk import LabelStudio
import httpx
import pandas as pd

load_dotenv()

True

## Load the data

In [3]:
CSV_PATH = "/gpfs1/home/j/s/jstonge1/rural-geog-classif/extract/input/Full Dataset Rur Geog WoS 1986-2025 4-28-2026.csv"

df = pd.read_csv(CSV_PATH, usecols=["Article Title", "Abstract", "DOI", "Pub Year", "Authors"], encoding='latin')
df = df.rename(columns={
    "Article Title": "title",
    "Abstract": "abstract",
    "DOI": "doi",
    "Pub Year": "year",
    "Authors": "authors",
})

before = len(df)
df = df.dropna(subset=["title", "abstract"])
df = df[df["abstract"].str.strip().astype(bool)]
print(f"Loaded {len(df)} papers with title+abstract (dropped {before - len(df)} missing)")

df.head(3)[["doi", "year", "authors", "title"]]

Loaded 360 papers with title+abstract (dropped 10 missing)


,doi,year,authors,title
0,10.1111/gere.12244,2018,"Abizaid, C; Coomes, OT; Takasaki, Y; Arroyo-Mo...",Rural Social Networks along Amazonian Rivers: ...
1,10.1080/00330124.2025.2582789,2025,"Adu-Poku, A; Appiah, IG; Kemausuor, F",Geographical Disparities in Energy Access: Cha...
2,10.1080/00045608.2014.985626,2015,"Aguayo, BC; Latta, A",Agro-Ecology and Food Sovereignty Movements in...


## Run base model

In [4]:
topic_model = BERTopic()

In [5]:
topics, probs = topic_model.fit_transform(df.abstract)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,148,-1_the_and_of_in,"[the, and, of, in, to, that, this, rural, for,...",[The South African homelands were central to t...
1,0,74,0_the_of_and_in,"[the, of, and, in, to, that, on, rural, by, as]",[This article examines the formation of social...
2,1,38,1_and_the_of_rural,"[and, the, of, rural, in, to, as, that, for, t...",[Rural geography has gone through profound cha...
3,2,21,2_the_in_and_of,"[the, in, and, of, migration, to, de, west, th...",[One of the most recognizable and important ch...
4,3,19,3_the_and_in_of,"[the, and, in, of, to, resilience, for, urban,...",[A composite of cloud-free radiance-calibrated...
5,4,17,4_the_of_and_to,"[the, of, and, to, in, land, areas, de, urban,...",[Environmental history in the Midwestern Corn ...
6,5,15,5_the_of_and_in,"[the, of, and, in, to, rural, farms, land, was...",[Changes in the nature of capitalist productio...
7,6,15,6_of_the_and_in,"[of, the, and, in, to, that, as, migration, on...","[Beginning with Friedrich Ratzel, the founders..."
8,7,13,7_and_of_the_in,"[and, of, the, in, development, to, china, on,...",[The tremendous changes in China's development...


In [7]:
topic_model.get_topic(1)

[('and', np.float64(0.06572668446193176)),
 ('the', np.float64(0.06569352950453788)),
 ('of', np.float64(0.06338493770939979)),
 ('rural', np.float64(0.04459793145156901)),
 ('in', np.float64(0.042341013950568246)),
 ('to', np.float64(0.04124747208746554)),
 ('as', np.float64(0.03091098157036583)),
 ('that', np.float64(0.02765305684507914)),
 ('for', np.float64(0.02451019800815956)),
 ('this', np.float64(0.02390143530079395))]

The topics don't look promising, lets move to the next thing.

## Fine-tune topic representations

In [8]:
from bertopic.representation import KeyBERTInspired

In [9]:
representation_model = KeyBERTInspired()
topic_model = BERTopic(representation_model=representation_model)

In [10]:
topics, probs = topic_model.fit_transform(df.abstract)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,122,-1_rural_agricultural_farmers_land,"[rural, agricultural, farmers, land, communiti...",[This paper argues against the dichotomization...
1,0,96,0_rural_landscape_geography_communities,"[rural, landscape, geography, communities, eco...","[Over the past two decades, many of Mexico's r..."
2,1,82,1_livelihoods_livelihood_rural_sovereignty,"[livelihoods, livelihood, rural, sovereignty, ...",[Developing the Amazon into a major provider o...
3,2,18,2_rural_poverty_socioeconomic_geographically,"[rural, poverty, socioeconomic, geographically...",[In this study we analyzed the diurnal spatial...
4,3,17,3_farmland_rural_lands_land,"[farmland, rural, lands, land, habitat, regime...",[Traditional approaches to studying human-envi...
5,4,14,4_agricultural_farmland_farming_farms,"[agricultural, farmland, farming, farms, lands...","[In his seminal work on cartography, Brian Har..."
6,5,11,5_rural_villages_ecology_ciudades,"[rural, villages, ecology, ciudades, environme...",[This paper analyzes environmental degradation...


In [12]:
topic_model.get_topic(1)

[('livelihoods', np.float32(0.47395572)),
 ('livelihood', np.float32(0.4546225)),
 ('rural', np.float32(0.39809087)),
 ('sovereignty', np.float32(0.38507968)),
 ('households', np.float32(0.37592468)),
 ('agriculture', np.float32(0.3710546)),
 ('farmers', np.float32(0.36932498)),
 ('agricultural', np.float32(0.36422282)),
 ('colonial', np.float32(0.3340168)),
 ('land', np.float32(0.33184618))]

Those topics already look better.

In [13]:
topic_model.get_document_info(df.abstract)

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,Research on Amazonian communities has focussed...,1,1_livelihoods_livelihood_rural_sovereignty,"[livelihoods, livelihood, rural, sovereignty, ...",[Developing the Amazon into a major provider o...,livelihoods - livelihood - rural - sovereignty...,1.000000,False
1,This study examines geographical disparities i...,-1,-1_rural_agricultural_farmers_land,"[rural, agricultural, farmers, land, communiti...",[This paper argues against the dichotomization...,rural - agricultural - farmers - land - commun...,0.000000,False
2,The agro-ecology and food sovereignty movement...,1,1_livelihoods_livelihood_rural_sovereignty,"[livelihoods, livelihood, rural, sovereignty, ...",[Developing the Amazon into a major provider o...,livelihoods - livelihood - rural - sovereignty...,1.000000,False
3,This article aims to advance understandings of...,1,1_livelihoods_livelihood_rural_sovereignty,"[livelihoods, livelihood, rural, sovereignty, ...",[Developing the Amazon into a major provider o...,livelihoods - livelihood - rural - sovereignty...,1.000000,False
4,Land change in the Amazon is driven by numerou...,1,1_livelihoods_livelihood_rural_sovereignty,"[livelihoods, livelihood, rural, sovereignty, ...",[Developing the Amazon into a major provider o...,livelihoods - livelihood - rural - sovereignty...,1.000000,False
...,...,...,...,...,...,...,...,...
355,The majority of the ancient rock art sites of ...,-1,-1_rural_agricultural_farmers_land,"[rural, agricultural, farmers, land, communiti...",[This paper argues against the dichotomization...,rural - agricultural - farmers - land - commun...,0.000000,False
356,This study examined the use of urban and rural...,3,3_farmland_rural_lands_land,"[farmland, rural, lands, land, habitat, regime...",[Traditional approaches to studying human-envi...,farmland - rural - lands - land - habitat - re...,0.994488,False
357,This article evaluates the effect of moving wi...,3,3_farmland_rural_lands_land,"[farmland, rural, lands, land, habitat, regime...",[Traditional approaches to studying human-envi...,farmland - rural - lands - land - habitat - re...,1.000000,False
358,Carbon accounting is an important analytical t...,-1,-1_rural_agricultural_farmers_land,"[rural, agricultural, farmers, land, communiti...",[This paper argues against the dichotomization...,rural - agricultural - farmers - land - commun...,0.000000,False


In [14]:
topic_model.visualize_topics()

## Follow best practices

https://maartengr.github.io/BERTopic/getting_started/best_practices/best_practices.html#pre-calculate-embeddings

In [19]:
from sentence_transformers import SentenceTransformer

# Pre-calculate embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(df.abstract.to_list(), show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

In [20]:
from umap import UMAP

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

In [25]:
# There is a parameter to control the number of topics, namely nr_topics. 
# This parameter, however, merges topics after they have been created. It is a parameter that supports creating a fixed number of topics.

# However, it is advised to control the number of topics through the cluster model which is by default HDBSCAN. HDBSCAN has a parameter, 
# namely min_cluster_size that indirectly controls the number of topics that will be created.

# A higher min_cluster_size will generate fewer topics and a lower min_cluster_size will generate more topics.

# Here, we will go with min_cluster_size=150 to prevent too many micro-clusters from being created:
from hdbscan import HDBSCAN

hdbscan_model = HDBSCAN(min_cluster_size=9, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

In [26]:
# Improving Default Representation¶
from sklearn.feature_extraction.text import CountVectorizer
vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))

In [27]:
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI, PartOfSpeech

# KeyBERT
keybert_model = KeyBERTInspired()

# Part-of-Speech
pos_model = PartOfSpeech("en_core_web_sm")

# MMR
mmr_model = MaximalMarginalRelevance(diversity=0.3)

# All representation models
representation_model = {
    "KeyBERT": keybert_model,
    "MMR": mmr_model,
    "POS": pos_model
}

In [28]:
# training
from bertopic import BERTopic

topic_model = BERTopic(

  # Pipeline models
  embedding_model=embedding_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  representation_model=representation_model,

  # Hyperparameters
  top_n_words=10,
  verbose=True
)

# Train model
topics, probs = topic_model.fit_transform(df.abstract.to_list(), embeddings)

# Show topics
topic_model.get_topic_info()

2026-05-11 14:33:13,800 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-11 14:33:14,232 - BERTopic - Dimensionality - Completed ✓
2026-05-11 14:33:14,232 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-11 14:33:14,239 - BERTopic - Cluster - Completed ✓
2026-05-11 14:33:14,240 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-11 14:33:25,563 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,111,-1_rural_article_urban_study,"[rural, article, urban, study, spatial, local,...","[rural areas, rural, geographic, geographical,...","[rural, local, areas, data, energy, communitie...","[rural, article, urban, study, spatial, local,...",[Divorce and family dissolution are global iss...
1,0,69,0_rural_development_water_political,"[rural, development, water, political, state, ...","[livelihoods, farmers, livelihood, agriculture...","[rural, development, land, governance, farmers...","[rural, development, water, political, state, ...",[Ethnic minority households in upland northern...
2,1,68,1_rural_urban_place_migration,"[rural, urban, place, migration, social, artic...","[rurality, rural communities, rural urban, rur...","[rural, place, migration, communities, rural u...","[rural, urban, place, migration, social, artic...",[Increasing numbers of young people are migrat...
3,2,45,2_land_china_urban_development,"[land, china, urban, development, rural, areas...","[land use, urban areas, urban rural, land, lan...","[china, urban, rural, areas, land use, growth,...","[land, urban, development, rural, areas, use, ...",[A deficiency common to both the historical de...
4,3,22,3_migration_west_rural_population,"[migration, west, rural, population, la, areas...","[rural, migration, emigration, migrants, popul...","[migration, rural, population, counties, emplo...","[migration, rural, population, areas, employme...","[Arguably, rural land markets in Australia are..."
5,4,19,4_urban_heat_resilience_areas,"[urban, heat, resilience, areas, la, poverty, ...","[urban areas, evacuation, rural areas, geograp...","[urban, heat, resilience, poverty, evacuation,...","[urban, heat, resilience, areas, poverty, evac...",[The concept of disaster resilience has gained...
6,5,14,5_land_farms_rural_changes,"[land, farms, rural, changes, eu, eastern, aba...","[farmland, land use, land reform, changes land...","[farms, rural, abandonment, housing, farmland,...","[land, farms, rural, changes, eastern, abandon...",[Changes in the nature of capitalist productio...
7,6,12,6_health_hiv_art_accessibility,"[health, hiv, art, accessibility, social, acce...","[rural, africa, hiv, botswana, hiv aids, aids,...","[health, hiv, accessibility, transit, faciliti...","[health, accessibility, social, access, spatia...",[This study presents a case study of how socia...


In [29]:
topic_model.get_topic(1, full=True)

{'Main': [('rural', np.float64(0.05604810120904597)),
  ('urban', np.float64(0.031936989205396435)),
  ('place', np.float64(0.028811008068900003)),
  ('migration', np.float64(0.026255621196154256)),
  ('social', np.float64(0.024419701194814677)),
  ('article', np.float64(0.022609901715985377)),
  ('people', np.float64(0.020479736136746234)),
  ('young', np.float64(0.020073594145590526)),
  ('development', np.float64(0.01965892491965975)),
  ('research', np.float64(0.019289707746401947))],
 'KeyBERT': [('rurality', np.float32(0.705014)),
  ('rural communities', np.float32(0.6621402)),
  ('rural urban', np.float32(0.63380873)),
  ('rural', np.float32(0.6283351)),
  ('rural areas', np.float32(0.60834277)),
  ('gentrification', np.float32(0.44861993)),
  ('geography', np.float32(0.4442961)),
  ('geographical', np.float32(0.4424794)),
  ('village', np.float32(0.41830334)),
  ('landscape', np.float32(0.38781852))],
 'MMR': [('rural', np.float64(0.05604810120904597)),
  ('place', np.float64(0

In [30]:
# `topic_distr` contains the distribution of topics in each document
topic_distr, _ = topic_model.approximate_distribution(df.abstract.to_list(), window=8, stride=4)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.88it/s]


In [39]:
from textwrap import wrap
abstract_id = 10
print('\n'.join(wrap(df.abstract.to_list()[abstract_id], 150)))
topic_model.visualize_distribution(topic_distr[abstract_id], custom_labels=True)

This article interrogates the concept of technical memory in relation to smart city systems. Using the example of the UK air pollution monitoring
system Automatic Urban and Rural Network (AURN) and how information from this system is displayed in smartphone air monitoring apps, the article
theorizes the memory of smart systems. Developing the work of Garcia, the article rethinks Stiegler's retentional accounts of technical memory, which
suggest that memory is held or inscribed on or within a particular technical object. To do this it argues that technical memory can be productively
considered as a form of artificial comprehension. Here, the memory of smart systems is analyzed through a variety of logics that disclose particular
qualities of objects for particular purposes, which shapes how people make sense of and respond to their environment. Through the example of AURN, the
article suggests that the concept of artificial comprehension is useful for geographers studying a range of sma

In [41]:
# Visualize topics with custom labels
topic_model.visualize_topics(custom_labels=True)

# Visualize hierarchy with custom labels
topic_model.visualize_hierarchy(custom_labels=True)

## Dynamic Topic model

https://maartengr.github.io/BERTopic/getting_started/topicsovertime/topicsovertime.html#example

In [ ]:
from bertopic import BERTopic

# using kerBert
representation_model = KeyBERTInspired()
topic_model = BERTopic(representation_model=representation_model, verbose=True)
topics, probs = topic_model.fit_transform(df.abstract)

2026-05-11 14:46:35,818 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

2026-05-11 14:46:40,451 - BERTopic - Embedding - Completed ✓
2026-05-11 14:46:40,451 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-11 14:46:40,604 - BERTopic - Dimensionality - Completed ✓
2026-05-11 14:46:40,604 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-11 14:46:40,611 - BERTopic - Cluster - Completed ✓
2026-05-11 14:46:40,613 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-11 14:46:40,931 - BERTopic - Representation - Completed ✓


In [57]:
bins = [1985, 1995, 2005, 2015, 2027]  
labels = [1985, 1995, 2005, 2015]
timestamps = pd.cut(df.year, bins=bins, labels=labels, right=False).astype(int).tolist()


topics_over_time = topic_model.topics_over_time(df.abstract.to_list(), timestamps)
topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=10)

4it [00:10,  2.63s/it]
